# Fact and Dimension Tables

## Overview
This notebook covers querying and loading the star schema's two building blocks: fact tables (what happened, with measures) and dimension tables (who/what/when/where).

**Loading order rule:** always load dimensions before facts, because fact rows reference dimension surrogate keys. For every dimension, insert a `-1 / 'Unknown'` member first to handle unresolved foreign key lookups.

```
  dim_date → dim_customer → dim_product → dim_agent → fact_policy
```

---

In [1]:
import sqlite3, json, random
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.executescript("""
-- ── Dimension tables ──────────────────────────────────────────────
CREATE TABLE dim_date (
    date_key    INTEGER PRIMARY KEY,  -- YYYYMMDD
    full_date   TEXT NOT NULL,
    year        INTEGER, quarter INTEGER, month INTEGER,
    month_name  TEXT,    week_of_year INTEGER, day_of_week INTEGER,
    day_name    TEXT,    is_weekend INTEGER, is_holiday INTEGER
);

CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id  TEXT NOT NULL,
    full_name    TEXT, province TEXT, city TEXT,
    segment      TEXT,  -- 'retail','commercial','group'
    risk_tier    TEXT,  -- 'low','medium','high'
    effective_from TEXT, effective_to TEXT, is_current INTEGER DEFAULT 1
);

CREATE TABLE dim_product (
    product_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id   TEXT NOT NULL,
    product_name TEXT,
    product_line TEXT,  -- 'auto','home','life','health'
    category     TEXT,
    premium_band TEXT   -- 'basic','standard','premium'
);

CREATE TABLE dim_agent (
    agent_key    INTEGER PRIMARY KEY AUTOINCREMENT,
    agent_id     TEXT NOT NULL,
    agent_name   TEXT,
    region       TEXT,
    channel      TEXT   -- 'direct','broker','online'
);

-- ── Fact table ────────────────────────────────────────────────────
CREATE TABLE fact_policy (
    policy_key      INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key        INTEGER REFERENCES dim_date(date_key),
    customer_key    INTEGER REFERENCES dim_customer(customer_key),
    product_key     INTEGER REFERENCES dim_product(product_key),
    agent_key       INTEGER REFERENCES dim_agent(agent_key),
    -- Measures
    premium_amount  REAL,
    coverage_amount REAL,
    claim_amount    REAL DEFAULT 0,
    n_claims        INTEGER DEFAULT 0,
    policy_count    INTEGER DEFAULT 1
);

-- ── Staging table (ETL landing zone) ─────────────────────────────
CREATE TABLE stg_policy_load (
    src_policy_id   TEXT,
    customer_id     TEXT,
    product_id      TEXT,
    agent_id        TEXT,
    policy_date     TEXT,
    premium         REAL,
    coverage        REAL,
    load_ts         TEXT DEFAULT (datetime('now')),
    load_status     TEXT DEFAULT 'pending'
);
""")

# Populate dim_date (2022-2024)
dates = []
d = date(2022, 1, 1)
while d <= date(2024, 12, 31):
    dk = int(d.strftime('%Y%m%d'))
    mnames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    dnames = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    dates.append((dk, str(d), d.year, (d.month-1)//3+1, d.month,
                  mnames[d.month-1], d.isocalendar()[1], d.weekday(),
                  dnames[d.weekday()], 1 if d.weekday()>=5 else 0, 0))
    d += timedelta(days=1)
conn.executemany("INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?,?,?,?)", dates)

# Populate dimensions
customers = [
    (1,'C001','Alice Nguyen','ON','Toronto','retail','low','2022-01-01',None,1),
    (2,'C002','Bob Harrington','BC','Vancouver','commercial','medium','2022-01-01',None,1),
    (3,'C003','Fatima Al-Rashid','QC','Montreal','group','low','2022-01-01',None,1),
    (4,'C004','James MacLeod','NS','Halifax','retail','high','2022-01-01',None,1),
    (5,'C005','Mei-Ling Chen','AB','Calgary','commercial','medium','2022-01-01',None,1),
    (6,'C006','David Park','ON','Ottawa','retail','low','2023-06-01',None,1),
]
conn.executemany("INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?,?,?,?)", customers)

products = [
    (1,'P01','Auto Basic','auto','liability','basic'),
    (2,'P02','Auto Comprehensive','auto','comprehensive','standard'),
    (3,'P03','Home Standard','home','dwelling','standard'),
    (4,'P04','Home Premium','home','dwelling','premium'),
    (5,'P05','Term Life 10','life','term','basic'),
    (6,'P06','Whole Life','life','permanent','premium'),
    (7,'P07','Group Health','health','group','standard'),
]
conn.executemany("INSERT INTO dim_product VALUES (?,?,?,?,?,?)", products)

agents = [
    (1,'A01','Sandra Lee','East','direct'),
    (2,'A02','Tom Okafor','West','broker'),
    (3,'A03','Priya Sharma','Central','online'),
    (4,'A04','Marc Tremblay','East','broker'),
]
conn.executemany("INSERT INTO dim_agent VALUES (?,?,?,?,?)", agents)

# Generate fact_policy rows (2022-2024)
random.seed(42)
facts = []
policy_dates = [
    (20220115,20220315,20220601,20220901,20221201),
    (20230115,20230401,20230715,20231001,20231215),
    (20240201,20240401,20240701,20241001,20241201),
]
combos = [
    (20220115,1,1,1,1200,50000,0,0),
    (20220315,1,3,2,1800,180000,0,0),
    (20220601,2,2,1,2400,120000,500,1),
    (20220901,3,5,3,600,200000,0,0),
    (20221201,4,1,4,1400,60000,2200,1),
    (20230115,1,2,2,2600,130000,0,0),
    (20230401,2,4,1,3200,250000,0,0),
    (20230715,5,7,3,4800,0,0,0),
    (20231001,6,3,2,2100,195000,1500,1),
    (20231215,3,6,4,5400,500000,0,0),
    (20240201,4,2,1,1600,70000,800,1),
    (20240401,1,5,3,700,220000,0,0),
    (20240701,5,4,2,3500,280000,0,0),
    (20241001,2,1,1,1100,55000,0,0),
    (20241201,6,7,3,5200,0,0,0),
    (20220115,3,3,2,2000,200000,600,1),
    (20230601,4,4,1,3800,300000,0,0),
    (20240301,5,2,4,2200,115000,0,0),
]
for i, (dk,ck,pk,ak,prem,cov,claim,nclaim) in enumerate(combos):
    facts.append((dk,ck,pk,ak,prem,cov,claim,nclaim,1))
conn.executemany(
    "INSERT INTO fact_policy (date_key,customer_key,product_key,agent_key,premium_amount,coverage_amount,claim_amount,n_claims,policy_count) VALUES (?,?,?,?,?,?,?,?,?)",
    facts)
conn.commit()

print("Data warehouse dataset loaded:")
for t in ['dim_date','dim_customer','dim_product','dim_agent','fact_policy']:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22s}: {n} rows")

# Quick sanity check
total = conn.execute("SELECT SUM(premium_amount) FROM fact_policy").fetchone()[0]
print(f"\n  Total premiums: ${total:,.0f}")


Data warehouse dataset loaded:
  dim_date              : 1096 rows
  dim_customer          : 6 rows
  dim_product           : 7 rows
  dim_agent             : 4 rows
  fact_policy           : 18 rows

  Total premiums: $45,600


---
## Querying the star schema

In [2]:
print("=== Querying fact + dimension tables ===")
print()

# Basic star schema join
rows = conn.execute("""
    SELECT dd.year, dd.month_name,
           dp.product_line,
           da.region,
           SUM(fp.premium_amount)  AS total_premium,
           SUM(fp.claim_amount)    AS total_claims,
           COUNT(fp.policy_count)  AS n_policies,
           ROUND(100.0 * SUM(fp.claim_amount) / NULLIF(SUM(fp.premium_amount),0), 1) AS loss_ratio_pct
    FROM   fact_policy fp
    JOIN   dim_date     dd ON dd.date_key     = fp.date_key
    JOIN   dim_product  dp ON dp.product_key  = fp.product_key
    JOIN   dim_agent    da ON da.agent_key    = fp.agent_key
    GROUP  BY dd.year, dd.month_name, dp.product_line, da.region
    HAVING SUM(fp.premium_amount) > 0
    ORDER  BY dd.year, SUM(fp.premium_amount) DESC
    LIMIT  12
""").fetchall()

print("Premium and loss ratio by year / product line / region:")
print(f"  {'year':<6s}  {'month':<6s}  {'product_line':<10s}  {'region':<8s}  {'premium':>10s}  {'claims':>10s}  {'n_pol':>6s}  {'loss%':>6s}")
print("  " + "-"*68)
for r in rows:
    print(f"  {r['year']:<6d}  {r['month_name']:<6s}  {r['product_line']:<10s}  {r['region']:<8s}  "
          f"${r['total_premium']:>9,.0f}  ${r['total_claims']:>9,.0f}  {r['n_policies']:>6d}  "
          f"{r['loss_ratio_pct'] or 0:>5.1f}%")


=== Querying fact + dimension tables ===

Premium and loss ratio by year / product line / region:
  year    month   product_line  region       premium      claims   n_pol   loss%
  --------------------------------------------------------------------
  2022    Jun     auto        East      $    2,400  $      500       1   20.8%
  2022    Jan     home        West      $    2,000  $      600       1   30.0%
  2022    Mar     home        West      $    1,800  $        0       1    0.0%
  2022    Dec     auto        East      $    1,400  $    2,200       1  157.1%
  2022    Jan     auto        East      $    1,200  $        0       1    0.0%
  2022    Sep     life        Central   $      600  $        0       1    0.0%
  2023    Dec     life        East      $    5,400  $        0       1    0.0%
  2023    Jul     health      Central   $    4,800  $        0       1    0.0%
  2023    Jun     home        East      $    3,800  $        0       1    0.0%
  2023    Apr     home        East     

---
## dim_date: the calendar dimension

In [3]:
print("=== dim_date: the calendar dimension ===")
print()

# Show the power of dim_date for period filtering
rows = conn.execute("""
    SELECT dd.year, dd.quarter,
           SUM(fp.premium_amount) AS total_premium,
           SUM(fp.n_claims)       AS total_claims
    FROM   fact_policy fp
    JOIN   dim_date dd ON dd.date_key = fp.date_key
    GROUP  BY dd.year, dd.quarter
    ORDER  BY dd.year, dd.quarter
""").fetchall()
print("Premium by year/quarter (using dim_date hierarchy):")
for r in rows:
    bar = '█' * int(r['total_premium']//500)
    print(f"  {r['year']} Q{r['quarter']}  ${r['total_premium']:>8,.0f}  {bar}")

print()
print("dim_date columns and usage:")
dd_cols = [
    ("date_key",     "INTEGER YYYYMMDD",  "FK from fact tables — faster than DATE join"),
    ("full_date",    "TEXT date",          "For human-readable display"),
    ("year",         "INTEGER",           "GROUP BY year"),
    ("quarter",      "INTEGER 1-4",       "GROUP BY quarter; period comparisons"),
    ("month",        "INTEGER 1-12",      "GROUP BY month"),
    ("month_name",   "TEXT",              "Display label: 'Jan','Feb'..."),
    ("week_of_year", "INTEGER 1-53",      "Weekly aggregations"),
    ("day_of_week",  "INTEGER 0=Mon",     "Weekday analysis"),
    ("is_weekend",   "INTEGER 0/1",       "WHERE is_weekend = 0 for business days"),
    ("is_holiday",   "INTEGER 0/1",       "Exclude holidays from business-day counts"),
]
print(f"  {'column':<14s}  {'type':<18s}  Usage")
print("  " + "-"*60)
for col, typ, use in dd_cols:
    print(f"  {col:<14s}  {typ:<18s}  {use}")

print()
print("PostgreSQL: generate dim_date in one query:")
print("""
  INSERT INTO dim_date
  SELECT TO_CHAR(d, 'YYYYMMDD')::INT  AS date_key,
         d::DATE                       AS full_date,
         EXTRACT(YEAR  FROM d)::INT    AS year,
         EXTRACT(QUARTER FROM d)::INT  AS quarter,
         EXTRACT(MONTH FROM d)::INT    AS month,
         TO_CHAR(d, 'Mon')             AS month_name,
         EXTRACT(WEEK  FROM d)::INT    AS week_of_year,
         EXTRACT(DOW   FROM d)::INT    AS day_of_week,
         TO_CHAR(d, 'Dy')              AS day_name,
         CASE WHEN EXTRACT(DOW FROM d) IN (0,6) THEN 1 ELSE 0 END AS is_weekend,
         0                             AS is_holiday
  FROM   GENERATE_SERIES('2020-01-01'::DATE, '2030-12-31'::DATE, '1 day') AS d;
""")


=== dim_date: the calendar dimension ===

Premium by year/quarter (using dim_date hierarchy):
  2022 Q1  $   5,000  ██████████
  2022 Q2  $   2,400  ████
  2022 Q3  $     600  █
  2022 Q4  $   1,400  ██
  2023 Q1  $   2,600  █████
  2023 Q2  $   7,000  ██████████████
  2023 Q3  $   4,800  █████████
  2023 Q4  $   7,500  ███████████████
  2024 Q1  $   3,800  ███████
  2024 Q2  $     700  █
  2024 Q3  $   3,500  ███████
  2024 Q4  $   6,300  ████████████

dim_date columns and usage:
  column          type                Usage
  ------------------------------------------------------------
  date_key        INTEGER YYYYMMDD    FK from fact tables — faster than DATE join
  full_date       TEXT date           For human-readable display
  year            INTEGER             GROUP BY year
  quarter         INTEGER 1-4         GROUP BY quarter; period comparisons
  month           INTEGER 1-12        GROUP BY month
  month_name      TEXT                Display label: 'Jan','Feb'...
  week_of_ye

---
## Loading facts from staging

In [4]:
print("=== Loading fact tables from staging ===")
print()

# Insert staging data
conn.execute("""
    INSERT INTO stg_policy_load (src_policy_id, customer_id, product_id, agent_id, policy_date, premium, coverage)
    VALUES
        ('POL-9001','C001','P03','A02','2024-11-15',2200,210000),
        ('POL-9002','C005','P01','A01','2024-11-20',1050,52000),
        ('POL-9003','C999','P02','A03','2024-11-25',1900,100000)  -- C999 unknown customer
""")
conn.commit()

print("Staging rows to load:")
rows = conn.execute("SELECT * FROM stg_policy_load").fetchall()
for r in rows:
    print(f"  {r['src_policy_id']}  customer={r['customer_id']}  product={r['product_id']}  premium={r['premium']}")

print()
# Load into fact table via dimension lookups
conn.execute("""
    INSERT INTO fact_policy (date_key, customer_key, product_key, agent_key,
                             premium_amount, coverage_amount, policy_count)
    SELECT
        CAST(REPLACE(s.policy_date, '-', '') AS INTEGER) AS date_key,
        COALESCE(c.customer_key, -1)  AS customer_key,
        COALESCE(p.product_key,  -1)  AS product_key,
        COALESCE(a.agent_key,    -1)  AS agent_key,
        s.premium,
        s.coverage,
        1
    FROM   stg_policy_load s
    LEFT JOIN dim_customer c ON c.customer_id = s.customer_id AND c.is_current = 1
    LEFT JOIN dim_product  p ON p.product_id  = s.product_id
    LEFT JOIN dim_agent    a ON a.agent_id    = s.agent_id
    WHERE  s.load_status = 'pending'
""")
conn.execute("UPDATE stg_policy_load SET load_status = 'loaded'")
conn.commit()

loaded = conn.execute("SELECT COUNT(*) FROM fact_policy").fetchone()[0]
print(f"fact_policy rows after load: {loaded}")

# Check the -1 surrogate for unknown customer
unk = conn.execute("""
    SELECT fp.policy_key, fp.customer_key, fp.premium_amount
    FROM   fact_policy fp
    WHERE  fp.customer_key = -1
""").fetchall()
print(f"Policies with unknown customer (key=-1): {len(unk)}")
for r in unk:
    print(f"  policy_key={r['policy_key']}  premium=${r['premium_amount']:.0f}")


=== Loading fact tables from staging ===

Staging rows to load:
  POL-9001  customer=C001  product=P03  premium=2200.0
  POL-9002  customer=C005  product=P01  premium=1050.0
  POL-9003  customer=C999  product=P02  premium=1900.0

fact_policy rows after load: 21
Policies with unknown customer (key=-1): 1
  policy_key=21  premium=$1900


---
## Common Pitfalls

**1. Not inserting the -1 'Unknown' surrogate before loading facts**
If a fact row has no matching dimension member and the FK is NOT NULL, the INSERT fails. If FK is NULL, the fact is silently disconnected from dimensions. Always pre-insert a surrogate key -1 'Unknown' row in every dimension, and use `COALESCE(dim.key, -1)` in the fact load query.

**2. Loading the same fact rows twice (no deduplication)**
Re-running the ETL job without clearing staging causes duplicate facts. Use a `load_status = 'pending'` flag on staging rows and mark them 'loaded' after insert. Or use a unique constraint on a natural fact key (policy_id) in the fact table.

**3. Integer date_key mismatch (YYYYMMDD vs YYYYDDD vs epoch)**
Date keys can be YYYYMMDD (`20241201`), YYYYDDD (`2024336`), or Unix epoch. Mixing formats across source systems is a common ETL bug. Standardise on YYYYMMDD (human-readable) and validate on load.

**4. dim_date not covering the full fact date range**
If dim_date only covers 2022–2024 and a late-arriving fact has `date_key = 20250101`, the JOIN returns NULL and the fact is excluded from queries. Populate dim_date 5–10 years ahead of current date.

**5. Allowing NULL foreign keys in fact tables**
NULL FKs silently exclude facts from aggregations that join to the dimension. Enforce NOT NULL FKs with a -1 unknown surrogate rather than allowing NULLs.

**6. Confusing additive and non-additive measures**
SUM(premium_amount) across all dimensions is valid (additive). SUM(loss_ratio_pct) is not valid — you cannot sum ratios. Identify each measure's additivity type at design time and document it in the DWH metadata.


---
*sql_methods_library - Samantha McGarrigle*